# ♟️ Google Colab Stockfish 19 Dev/Master Analysis Server

This notebook builds the **Stockfish 19 Dev/Master** engine with CPU/GPU auto-detection, launches the FastAPI server backend, and exposes a public HTTPS tunnel via Cloudflare. Copy the public HTTPS URL into your Render web app settings to connect!

In [ ]:
# STEP 1 — Install System Dependencies and Python Libraries
!apt-get update -qq
!apt-get install -y -qq build-essential git python3 python3-pip curl
!pip install -q python-chess fastapi uvicorn cloudflared
print("✅ System dependencies installed successfully!")

In [ ]:
# STEP 2 — Detect CPU/GPU Capabilities & Build Stockfish 19 Dev/Master
import os
import subprocess

print("🔍 Detecting hardware architecture...")
cpu_flags = subprocess.getoutput("lscpu")
gpu_info = subprocess.getoutput("nvidia-smi")
arch = "x86-64-modern"

if "avx2" in cpu_flags.lower():
    arch = "x86-64-avx2"
if "avx512" in cpu_flags.lower():
    arch = "x86-64-avx512"

print(f"⚡ Selected Stockfish build ARCH={arch}")

if not os.path.exists("Stockfish"):
    print("📦 Cloning Stockfish master repository...")
    !git clone --depth 1 https://github.com/official-stockfish/Stockfish.git

%cd Stockfish/src
!make -j$(nproc) build ARCH={arch}
%cd /content

stockfish_path = "/content/Stockfish/src/stockfish"
if os.path.exists(stockfish_path):
    print("✅ Stockfish 19 Dev/Master build complete!")
    res = subprocess.getoutput(f"{stockfish_path} uci")
    print(res.splitlines()[0] if res else "UCI OK")
else:
    print("⚠️ Stockfish binary build error. Falling back to default system build.")

In [ ]:
# STEP 3 — Start FastAPI Server in Background
import os
import time
import subprocess
import requests

os.makedirs("/content/app", exist_ok=True)
if os.path.exists("/content/app/backend"):
    %cd /content/app

print("🚀 Starting FastAPI server on http://127.0.0.1:8000...")
server_process = subprocess.Popen(
    ["python3", "-m", "uvicorn", "backend.app:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

for _ in range(15):
    try:
        resp = requests.get("http://127.0.0.1:8000/api/health", timeout=2)
        if resp.status_code == 200:
            print("✅ FastAPI backend is active and healthy!")
            break
    except Exception:
        time.sleep(1)

In [ ]:
# STEP 4 — Launch Public Cloudflare Tunnel & Display Server URL
import time
import subprocess
import re

print("🌐 Creating Cloudflare Tunnel...")
tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for _ in range(30):
    line = tunnel_process.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

if public_url:
    print("\n========================================")
    print("CHESS ANALYSIS SERVER ONLINE")
    print("========================================")
    print(f"Local:  http://127.0.0.1:8000")
    print(f"Public: {public_url}")
    print("\nCopy this Public URL into your Render App -> Settings -> Colab Server URL")
    print("========================================\n")
else:
    print("⚠️ Could not automatically parse Cloudflare URL. Please check tunnel output.")

In [ ]:
# STEP 5 — Keep-Alive Heartbeat Loop
import time
import requests

print("💓 Heartbeat keep-alive active. Press Stop in Colab to terminate server.")
print("Note: Google Colab runtimes are temporary and will eventually disconnect when idle.")

while True:
    try:
        resp = requests.get("http://127.0.0.1:8000/api/health", timeout=5)
        print(f"[Heartbeat OK] Engine: {resp.json().get('engine')}")
    except Exception as e:
        print(f"[Heartbeat Warning] {e}")
    time.sleep(30)